In [ ]:
# ============================================================
#  Model Selection Notebook (Final – Clean Version)
# ============================================================

import os
import json
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
CURRENT_DATASET = "Flickr8k"

BASE_DIR = "/home/aysel/tfe/TFE_Data"
RESULTS_DIR = os.path.join(BASE_DIR, "Unimodal_Results", CURRENT_DATASET)

# ------------------------------------------------------------
# Load unimodal retrieval metrics
# ------------------------------------------------------------

vision_perf = pd.read_csv(os.path.join(RESULTS_DIR, "vision", "vision_retrieval_results.csv"))
text_perf   = pd.read_csv(os.path.join(RESULTS_DIR, "text", "text_retrieval_results.csv"))

# ------------------------------------------------------------
# Load explainability metrics
# ------------------------------------------------------------

exp_vision = pd.read_csv("/home/aysel/tfe/Explainability_Vision.csv")
exp_text   = pd.read_csv("/home/aysel/tfe/Explainability_Text.csv")

# ------------------------------------------------------------
# Load global efficiency metrics (pickle)
# ------------------------------------------------------------

eff_df = pd.read_pickle("TFE_Data/Unimodal_Results/global_unimodal_metrics.pkl")
eff_df = eff_df.rename(columns={"Model": "model"})

eff_vision = eff_df[eff_df["Modality"] == "vision"].copy()
eff_text   = eff_df[eff_df["Modality"] == "text"].copy()

# Rename efficiency columns
eff_vision = eff_vision.rename(columns={
    "Time_s": "embedding_time",
    "Latency_s": "inference_time",
    "Memory_MB": "memory_mb"
})

eff_text = eff_text.rename(columns={
    "Time_s": "embedding_time",
    "Latency_s": "inference_time",
    "Memory_MB": "memory_mb"
})

# ------------------------------------------------------------
# Normalize model names
# ------------------------------------------------------------

def normalize_model_names(df):
    df["model"] = (
        df["model"]
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace("-", "_", regex=False)
        .str.replace(" ", "_", regex=False)
    )
    return df

def minmax(series, higher_is_better=True):
    s_min = series.min()
    s_max = series.max()

    if s_min == s_max:
        return pd.Series([0.5] * len(series), index=series.index)

    norm = (series - s_min) / (s_max - s_min)

    if not higher_is_better:
        norm = 1 - norm

    return norm

def invert_cost(series):
    # 1/(1+x) avoids division by zero and keeps values in (0,1]
    return 1 / (1 + series)

def capitalize_columns(df):
    df.columns = [c.capitalize() for c in df.columns]
    return df

def clean_model_name(name):
    name = name.replace("_", " ")
    name = " ".join([w.capitalize() for w in name.split()])
    return name


vision_perf = normalize_model_names(vision_perf)
text_perf   = normalize_model_names(text_perf)
exp_vision  = normalize_model_names(exp_vision)
exp_text    = normalize_model_names(exp_text)
eff_vision  = normalize_model_names(eff_vision)
eff_text    = normalize_model_names(eff_text)

# ------------------------------------------------------------
# Merge metrics per modality
# ------------------------------------------------------------

vision_df = (
    vision_perf
    .merge(exp_vision, on="model", how="left")
    .merge(eff_vision, on="model", how="left")
)

text_df = (
    text_perf
    .merge(exp_text, on="model", how="left")
    .merge(eff_text, on="model", how="left")
)

# Drop useless columns
vision_df = vision_df.drop(columns=["Modality"], errors="ignore")
text_df   = text_df.drop(columns=["Modality"], errors="ignore")

# ------------------------------------------------------------
# Normalize only explainability + efficiency
# (NOT recall values)
# ------------------------------------------------------------

explain_cols     = ["faithfulness", "sparsity", "rank_corr", "complexity"]
efficiency_cols  = ["inference_time", "embedding_time", "memory_mb"]

def normalize_columns(df, columns):
    for col in columns:
        s = df[col].sum()
        if s == 0:
            df[col] = 0
        else:
            df[col] = df[col] / s
    return df

OUTPUT_DIR = "/home/aysel/tfe/TFE_Data/Unimodal_Results/Flickr8k"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Clean BEFORE-normalization model names
vision_perf_before = vision_perf.copy()
text_perf_before   = text_perf.copy()
exp_vision_before  = exp_vision.copy()
exp_text_before    = exp_text.copy()
eff_vision_before  = eff_vision.copy()
eff_text_before    = eff_text.copy()

for df in [vision_perf_before, text_perf_before, exp_vision_before, exp_text_before, eff_vision_before, eff_text_before]:
    df["model"] = df["model"].apply(clean_model_name)
    df = capitalize_columns(df)

# Save BEFORE-normalization
vision_perf_before.to_csv(f"{OUTPUT_DIR}/vision_performance.csv", index=False)
exp_vision_before.to_csv(f"{OUTPUT_DIR}/vision_explainability.csv", index=False)
eff_vision_before.to_csv(f"{OUTPUT_DIR}/vision_efficiency.csv", index=False)

text_perf_before.to_csv(f"{OUTPUT_DIR}/text_performance.csv", index=False)
exp_text_before.to_csv(f"{OUTPUT_DIR}/text_explainability.csv", index=False)
eff_text_before.to_csv(f"{OUTPUT_DIR}/text_efficiency.csv", index=False)



# No normalization needed
performance_cols = ["recall@1", "recall@5", "recall@10"]

explain_cols = ["faithfulness", "sparsity", "rank_corr", "complexity"]

for col in explain_cols:
    if col == "complexity":
        # lower is better → invert
        vision_df[col] = minmax(vision_df[col], higher_is_better=False)
        text_df[col]   = minmax(text_df[col],   higher_is_better=False)
    else:
        vision_df[col] = minmax(vision_df[col], higher_is_better=True)
        text_df[col]   = minmax(text_df[col],   higher_is_better=True)

eff_cols = ["inference_time", "embedding_time", "memory_mb"]

for col in eff_cols:
    vision_df[col] = invert_cost(vision_df[col])
    text_df[col]   = invert_cost(text_df[col])

# ------------------------------------------------------------
# Compute final scores
# ------------------------------------------------------------

def compute_scores(df):
    df["performance_score"] = (
        0.33 * df["recall@1"] +
        0.33 * df["recall@5"] +
        0.33 * df["recall@10"]
    )

    df["explainability_score"] = (
        0.25 * df["faithfulness"] +
        0.25 * df["sparsity"] +
        0.25 * df["rank_corr"] +
        0.25 * df["complexity"]
    )

    df["efficiency_score"] = (
        0.33 * df["inference_time"] +
        0.33 * df["embedding_time"] +
        0.33 * df["memory_mb"]
    )

    df["final_score"] = (
        0.33 * df["performance_score"] +
        0.33 * df["explainability_score"] +
        0.33 * df["efficiency_score"]
    )

compute_scores(vision_df)
compute_scores(text_df)

# ------------------------------------------------------------
# Sort each modality from best → worst
# ------------------------------------------------------------
# Clean model names
vision_df["model"] = vision_df["model"].apply(clean_model_name)
text_df["model"]   = text_df["model"].apply(clean_model_name)

# Drop unwanted columns
cols_to_drop = ["method", "scenario"]
vision_df = vision_df.drop(columns=cols_to_drop, errors="ignore")
text_df   = text_df.drop(columns=cols_to_drop, errors="ignore")

# Capitalize column names
vision_df = capitalize_columns(vision_df)
text_df   = capitalize_columns(text_df)

vision_df_sorted = vision_df.sort_values("Final_score", ascending=False)
text_df_sorted   = text_df.sort_values("Final_score", ascending=False)


# AFTER-normalization tables
vision_perf_after = vision_df[["Model", "Recall@1", "Recall@5", "Recall@10"]]
vision_expl_after = vision_df[["Model", "Faithfulness", "Sparsity", "Rank_corr", "Complexity"]]
vision_eff_after  = vision_df[["Model", "Inference_time", "Embedding_time", "Memory_mb"]]

text_perf_after = text_df[["Model", "Recall@1", "Recall@5", "Recall@10"]]
text_expl_after = text_df[["Model", "Faithfulness", "Sparsity", "Rank_corr", "Complexity"]]
text_eff_after  = text_df[["Model", "Inference_time", "Embedding_time", "Memory_mb"]]

# Save AFTER-normalization
vision_perf_after.to_csv(f"{OUTPUT_DIR}/vision_performance_normalized.csv", index=False)
vision_expl_after.to_csv(f"{OUTPUT_DIR}/vision_explainability_normalized.csv", index=False)
vision_eff_after.to_csv(f"{OUTPUT_DIR}/vision_efficiency_normalized.csv", index=False)

text_perf_after.to_csv(f"{OUTPUT_DIR}/text_performance_normalized.csv", index=False)
text_expl_after.to_csv(f"{OUTPUT_DIR}/text_explainability_normalized.csv", index=False)
text_eff_after.to_csv(f"{OUTPUT_DIR}/text_efficiency_normalized.csv", index=False)


vision_df_sorted


,Dataset,Model,Dim,Recall@1,Recall@5,Recall@10,Recall@50,Faithfulness,Complexity,Sparsity,...,Dataset,Num_samples,Embedding_time,Inference_time,Throughput_samples_per_s,Memory_mb,Performance_score,Explainability_score,Efficiency_score,Final_score
3,Flickr8k,Pvt,512,0.999876,1.0,1.0,1.0,0.000000,1.000000,1.000000,...,Flickr8k,8091,0.069391,0.998345,603.305909,0.930909,0.989959,0.750000,0.659553,0.791839
1,Flickr8k,Mobilenet V3,960,0.999876,1.0,1.0,1.0,0.379311,0.017622,0.123411,...,Flickr8k,8091,0.091144,0.998769,811.399165,1.000000,0.989959,0.228360,0.689671,0.629637
0,Flickr8k,Resnet50,2048,0.999876,1.0,1.0,1.0,0.503420,0.205473,0.336936,...,Flickr8k,8091,0.062846,0.998160,542.587043,0.351166,0.989959,0.261457,0.466017,0.566753
2,Flickr8k,Vit,768,0.999876,1.0,1.0,1.0,1.000000,0.000000,0.000000,...,Flickr8k,8091,0.031153,0.996171,260.167669,0.005953,0.989959,0.381966,0.340982,0.565259


In [2]:
text_df_sorted

,Dataset,Model,Dim,Recall@1,Recall@5,Recall@10,Recall@50,Faithfulness,Complexity,Sparsity,...,Dataset,Num_samples,Embedding_time,Inference_time,Throughput_samples_per_s,Memory_mb,Performance_score,Explainability_score,Efficiency_score,Final_score
1,Flickr8k,Roberta,768,0.993944,0.999852,1.0,1.0,1.00000,0.017123,0.000000,...,Flickr8k,40455,0.014052,0.998269,576.589154,0.044999,0.987953,0.504281,0.348916,0.607579
2,Flickr8k,Gpt2,768,0.993944,0.999852,1.0,1.0,0.00000,1.000000,0.489294,...,Flickr8k,40455,0.012187,0.998000,499.091334,0.000343,0.987953,0.425186,0.333475,0.576382
0,Flickr8k,Bert,768,0.993252,0.999802,1.0,1.0,0.03635,0.000000,1.000000,...,Flickr8k,40455,0.012672,0.998078,519.209158,0.000998,0.987708,0.259087,0.333877,0.521622


# Vision

In [11]:
VISION_MODELS = {
    "pvt": "",
    "vit": "",
    "resnet50": "",
    "mobilenet_v3": "",
}

from paths import (
    DATASETS_DIR,
    VISION_XAI_DIR,
    VISION_ARTIFACTS_DIR,
    VISION_EVAL_DIR,
    vision_xai_dir,
    vision_artifact_dir,
    vision_eval_dir,
    ensure_dirs
)

def load_metric_comparison_table():

    rows = []

    for model_name in VISION_MODELS.keys():

        summary_path = os.path.join(
            vision_xai_dir(CURRENT_DATASET, model_name),
            "metric_summary.csv"
        )

        if not os.path.exists(summary_path):
            print(f"Missing summary for {model_name}")
            continue

        # Load summary table
        summary = pd.read_csv(
            summary_path,
            index_col=0
        )

        row = {
            "model": model_name
        }

        # -----------------------------------------
        # SAFE METRIC EXTRACTION
        # -----------------------------------------
        metric_mapping = {
            "FaithfulnessCorrelation": "faithfulness",
            "Complexity": "complexity",
            "Sparseness": "sparsity",
            "rank_corr": "rank_corr",
        }

        for raw_metric, final_name in metric_mapping.items():

            if raw_metric in summary.index:

                row[final_name] = summary.loc[
                    raw_metric,
                    "mean"
                ]

            else:

                row[final_name] = np.nan

        rows.append(row)

    df_final = pd.DataFrame(rows)

    return df_final

In [12]:
import os
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
CURRENT_DATASET = "Flickr8k"

BASE_DIR = "/home/aysel/tfe"
VISION_DIR = f"TFE_Data/Evaluations/Unimodal/vision/{CURRENT_DATASET}"
TEXT_DIR = f"TFE_Data/Evaluations/Unimodal/text/{CURRENT_DATASET}"

v_retrieval_path = os.path.join( VISION_DIR, "vision_retrieval_results.csv")
v_eff_path = os.path.join(VISION_DIR,  "vision_efficiency_results.csv")

t_retrieval_path = os.path.join( TEXT_DIR, "text_retrieval_results.csv")
t_eff_path = os.path.join(TEXT_DIR,  "text_efficiency_results.csv")

V_OUTPUT_DIR = os.path.join(VISION_DIR, "model_selection")
os.makedirs(V_OUTPUT_DIR, exist_ok=True)

T_OUTPUT_DIR = os.path.join(TEXT_DIR, "model_selection")
os.makedirs(T_OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------
vision_perf = pd.read_csv(v_retrieval_path)
eff_vision  = pd.read_csv(v_eff_path)
exp_vision  = load_metric_comparison_table()


text_perf = pd.read_csv(t_retrieval_path)
eff_text  = pd.read_csv(t_eff_path)

In [13]:
vision_perf = vision_perf.drop(columns=["Dataset", "Dim"])
vision_perf

#vision_perf.to_csv(f"{V_OUTPUT_DIR}/vision_performance.csv", index=False)

,Model,Recall@1,Recall@5,Recall@10,Recall@50
0,ResNet50,0.999876,1.0,1.0,1.0
1,MobileNetV3,0.999876,1.0,1.0,1.0
2,ViT,0.999876,1.0,1.0,1.0
3,PVT,0.999876,1.0,1.0,1.0


In [14]:
eff_vision = eff_vision[eff_vision["Modality"] != "text"]
eff_vision = eff_vision.drop(columns=["Dataset", "Modality"])

eff_vision
#eff_vision.to_csv(f"{V_OUTPUT_DIR}/vision_efficiency.csv", index=False)



,Model,Num_Samples,Time_s,Latency_s,Throughput_samples_per_s,Memory_MB
0,ResNet50,8091,18.159502,0.002244,445.551871,3624.875000
1,MobileNetV3,8091,12.114647,0.001497,667.869225,8.515625
2,ViT,8091,60.184088,0.007438,134.437529,5844.355469
3,PVT,8091,19.957821,0.002467,405.404981,6.582031


In [15]:
exp_vision = exp_vision.rename(columns={
    "model": "Model",
    "faithfulness": "Faithfulness",
    "complexity": "Complexity",
    "sparsity": "Compactness",
    "rank_corr": "Rank_Corr"})

exp_vision["Model"] = exp_vision["Model"].replace({
    "pvt": "PVT",
    "vit": "ViT",
    "resnet50" : "ResNet50",
    "mobilenet_v3" : "MobileNetV3"
})

print(exp_vision.columns)
#exp_vision.to_csv(f"{V_OUTPUT_DIR}/vision_explainability.csv", index=False)

exp_vision

Index(['Model', 'Faithfulness', 'Complexity', 'Compactness', 'Rank_Corr'], dtype='str')


,Model,Faithfulness,Complexity,Compactness,Rank_Corr
0,PVT,-0.042926,9.989751,0.643774,0.004973
1,ViT,-0.009543,10.152931,0.593959,-0.025140
2,ResNet50,-0.026609,10.000115,0.644534,0.010911
3,MobileNetV3,-0.020717,10.081718,0.619473,0.002161


In [16]:
text_perf = text_perf.drop(columns=["dataset", "dim"])
text_perf = text_perf.rename(columns={
    "model": "Model",
    "recall@1": "Recall@1",
    "recall@5": "Recall@5",
    "recall@10": "Recall@10",
    "recall@50": "Recall@50"
    })
    
text_perf
text_perf.to_csv(f"{T_OUTPUT_DIR}/text_performance.csv")


In [17]:
eff_text = eff_text[eff_text["Modality"] != "vision"]
eff_text = eff_text.drop(columns=["Dataset", "Modality"])

eff_text = eff_text.rename(columns={
    "embedding_time": "Time_s",
    "inference_time": "Latency_s",
    "memory_mb": "Memory_MB",
    })

eff_text.to_csv(f"{T_OUTPUT_DIR}/text_efficiency.csv")


eff_text

,Model,Num_Samples,Time_s,Latency_s,Throughput_samples_per_s,Memory_MB
4,BERT,40455,142.056809,0.003511,284.780437,9643.156250
5,RoBERTa,40455,117.967974,0.002916,342.932058,4320.144531
6,GPT2,40455,137.787950,0.003406,293.603323,3639.523438


In [18]:
exp_text = pd.read_csv("text_explainability_results.csv")

exp_text = exp_text.rename(columns={
    "model": "Model",
    "faithfulness": "Faithfulness",
    "complexity": "Complexity",
    "sparsity": "Compactness",
    "paraphrase_robustness": "Paraphrase Robustness"})

exp_text.to_csv(f"{T_OUTPUT_DIR}/text_explainability.csv", index=False)

exp_text

,Model,Faithfulness,Complexity,Compactness,Paraphrase Robustness
0,BERT,-0.585570,2.428059,0.852363,0.626716
1,GPT2,0.049312,2.354109,0.868472,0.766264
2,RoBERTa,-0.324661,2.509093,0.839590,0.598739


In [22]:
soft_normalize(exp_text)

,Model,Faithfulness,Complexity,Compactness,Paraphrase Robustness
0,BERT,0.281769,0.507614,0.480796,0.397854
1,GPT2,0.741505,0.727985,0.738118,0.757787
2,RoBERTa,0.470509,0.266026,0.277007,0.326036


In [26]:
# ------------------------------------------------------------
# METRIC DIRECTION (CRITICAL)
# ------------------------------------------------------------
METRIC_DIRECTION = {
    "Faithfulness": "high",
    "Compactness": "high",
    "Rank_Corr": "high",
    "Complexity": "low",
    "Paragraph Robustness": "high",

    "Recall@1": "high",
    "Recall@5": "high",
    "Recall@10": "high",

    "Time_s": "low",
    "Latency_s": "low",
    "Throughput_samples_per_s": "high",
    "Memory_MB": "low",
}

# ------------------------------------------------------------
# SOFT NORMALIZATION 
# ------------------------------------------------------------
def soft_normalize(df):

    df_norm = df.copy()

    for col in df.columns:

        if col == "Model" or col == "Dataset":
            continue

        x = df[col].astype(float)

        # flip sign if lower is better
        if METRIC_DIRECTION.get(col, "high") == "low":
            x = -x

        z = (x - x.mean()) / (x.std() + 1e-8)

        df_norm[col] = 1 / (1 + np.exp(-z))
        

    return df_norm

def avg_normalize(df, columns):
    for col in columns:
        s = df[col].sum()
        if s == 0:
            df[col] = 0
        else:
            df[col] = df[col] / s
    return df

# ------------------------------------------------------------
# MERGE DATA
# ------------------------------------------------------------
vision_df = (
    vision_perf
    .merge(soft_normalize(exp_vision), on="Model", how="left")
    .merge(avg_normalize(eff_vision, ["Time_s", "Latency_s", "Throughput_samples_per_s", "Memory_MB"]), on="Model", how="left")
)

vision_df = vision_df.drop(columns=["Modality"], errors="ignore")


text_df = (
    text_perf
    .merge(soft_normalize(exp_text), on="Model", how="left")
    .merge(avg_normalize(eff_text, ["Time_s", "Latency_s", "Throughput_samples_per_s", "Memory_MB"]), on="Model", how="left")
)

text_df = text_df.drop(columns=["Modality"], errors="ignore")

# ------------------------------------------------------------
# SCORE FUNCTIONS (NOW CONSISTENT SCALE)
# ------------------------------------------------------------
def compute_scores_v(df):

    df["Performance_score"] = (
        0.33 * df["Recall@1"] +
        0.33 * df["Recall@5"] +
        0.33 * df["Recall@10"]
    )

    df["Explainability_score"] = (
        0.25 * df["Faithfulness"] +
        0.25 * df["Compactness"] +
        0.25 * df["Rank_Corr"] +
        0.25 *( 1-df["Complexity"])
    )

    df["Efficiency_score"] = (
        0.25 * (1-df["Time_s"]) +
        0.25 * (1-df["Latency_s"]) +
        0.25 * df["Throughput_samples_per_s"]+
        0.25 * (1-df["Memory_MB"])
    )

    df["Final_score"] = (
        0.33 * df["Performance_score"] +
        0.33 * df["Explainability_score"] +
        0.33 * df["Efficiency_score"]
    )

    return df

def compute_scores_t(df):

    df["Performance_score"] = (
        0.33 * df["Recall@1"] +
        0.33 * df["Recall@5"] +
        0.33 * df["Recall@10"]
    )

    df["Explainability_score"] = (
        0.25 * df["Faithfulness"] +
        0.25 * df["Compactness"] +
        0.25 * df["Paraphrase Robustness"] +
        0.25 *( 1-df["Complexity"])
    )

    df["Efficiency_score"] = (
        0.25 * (1-df["Time_s"]) +
        0.25 * (1-df["Latency_s"]) +
        0.25 * df["Throughput_samples_per_s"]+
        0.25 * (1-df["Memory_MB"])
    )

    df["Final_score"] = (
        0.33 * df["Performance_score"] +
        0.33 * df["Explainability_score"] +
        0.33 * df["Efficiency_score"]
    )

    return df

vision_df = compute_scores_v(vision_df)
text_df = compute_scores_t(text_df)

# ------------------------------------------------------------
# SORT MODELS
# ------------------------------------------------------------
vision_sorted = vision_df.sort_values(  "Final_score", ascending=False)
text_sorted = text_df.sort_values(  "Final_score", ascending=False)

# ------------------------------------------------------------
# SAVE CLEAN OUTPUT
# ------------------------------------------------------------
#vision_sorted.to_csv(os.path.join(V_OUTPUT_DIR, "vision_model_ranking.csv"), index=False)
text_sorted.to_csv(os.path.join(T_OUTPUT_DIR, "text_model_ranking.csv"), index=False)

In [27]:
text_sorted

,Model,Recall@1,Recall@5,Recall@10,Recall@50,Faithfulness,Complexity,Compactness,Paraphrase Robustness,Num_Samples,Time_s,Latency_s,Throughput_samples_per_s,Memory_MB,Performance_score,Explainability_score,Efficiency_score,Final_score
2,GPT2,0.993944,0.999852,1.0,1.0,0.741505,0.727985,0.738118,0.757787,40455,0.346364,0.346364,0.318678,0.206758,0.987953,0.627356,0.604798,0.732635
1,RoBERTa,0.993944,0.999852,1.0,1.0,0.470509,0.266026,0.277007,0.326036,40455,0.296541,0.296541,0.372220,0.245423,0.987953,0.451882,0.633428,0.684177
0,BERT,0.993252,0.999802,1.0,1.0,0.281769,0.507614,0.480796,0.397854,40455,0.357095,0.357095,0.309102,0.547819,0.987708,0.413201,0.511773,0.631185


In [20]:
vision_perf.to_csv(f"{V_OUTPUT_DIR}/vision_performance_normalized.csv", index=False)
soft_normalize(exp_vision).to_csv(f"{V_OUTPUT_DIR}/vision_explainability_normalized.csv", index=False)
avg_normalize(eff_vision, ["Time_s", "Latency_s", "Throughput_samples_per_s", "Memory_MB"]).to_csv(f"{V_OUTPUT_DIR}/vision_efficiency_normalized.csv", index=False)

print(f"{V_OUTPUT_DIR}/vision_efficiency_normalized.csv")

TFE_Data/Evaluations/Unimodal/vision/Flickr8k/model_selection/vision_efficiency_normalized.csv


In [21]:
text_perf.to_csv(f"{T_OUTPUT_DIR}/text_performance_normalized.csv", index=False)
soft_normalize(exp_text).to_csv(f"{T_OUTPUT_DIR}/text_explainability_normalized.csv", index=False)
avg_normalize(eff_text, ["Time_s", "Latency_s", "Throughput_samples_per_s", "Memory_MB"]).to_csv(f"{T_OUTPUT_DIR}/text_efficiency_normalized.csv", index=False)